# Traffic Anomaly Detection — Interactive Demo

Run this notebook to explore the project without installing anything locally.

1. Click **Open in Colab** in the README (or open this file from GitHub in Colab).
2. Run each cell in order (**Runtime → Run all**, or Shift+Enter per cell).
3. You will see synthetic traffic generation, processing, model training, and visualizations.

## 1. Setup (run once)

In [8]:
# Install dependencies (Colab has most; we add project-specific ones)
!pip install -q scikit-learn pandas numpy matplotlib plotly

import sys
from pathlib import Path

# If running in Colab after cloning the repo, project root is current dir
ROOT = Path('.')
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
else:
    # Allow running from notebook folder
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT.resolve())

Project root: /Users/nabilshehadeh/GITHUB REPO/trafficanomaly


## 2. Generate synthetic network traffic data

In [9]:
from src.data_collection.collector import NetworkDataCollector

collector = NetworkDataCollector(data_dir=str(ROOT / 'data'))
data = collector.generate_synthetic_data(2000)

print(f"Generated {len(data):,} samples")
print(f"Anomalies: {data['is_anomaly'].sum()} ({100*data['is_anomaly'].mean():.1f}%)")
print(f"Protocols: {list(data['protocol'].unique())}")
data.head(10)

INFO:src.data_collection.collector:Generating 2000 synthetic network traffic samples
INFO:src.data_collection.collector:Generated synthetic dataset with 2000 samples


Generated 2,000 samples
Anomalies: 98 (4.9%)
Protocols: ['ICMP', 'UDP', 'TCP']


,src_ip,dst_ip,src_port,dst_port,protocol,packet_length,duration,flow_bytes_sent,flow_bytes_received,packets_sent,packets_received,is_anomaly
0,192.168.103.180,10.0.178.161,51012,11999,ICMP,707,0.071176,8867,2751,38,9,0
1,192.168.93.15,10.0.219.7,60381,883,UDP,1423,0.098767,7080,639,20,67,0
2,192.168.107.72,10.0.70.127,44332,20225,ICMP,399,0.044066,779,6599,23,16,0
3,192.168.189.21,10.0.143.210,30498,39692,TCP,985,0.039080,3812,4855,37,89,0
4,192.168.103.122,10.0.160.248,31309,58215,ICMP,210,0.002342,9421,6059,62,88,0
5,192.168.211.215,10.0.203.148,55082,53764,TCP,185,0.069381,8609,3389,8,93,0
6,192.168.75.203,10.0.216.143,39561,60330,UDP,144,0.066089,5401,5646,37,55,0
7,192.168.88.117,10.0.176.199,53866,46477,TCP,748,0.017177,5834,2858,45,40,0
8,192.168.100.104,10.0.149.170,46687,58210,UDP,725,0.144530,8081,5411,97,26,0
9,192.168.152.131,10.0.154.236,10375,30826,UDP,438,0.067860,5992,4207,33,38,0


## 3. Process data (feature engineering)

In [10]:
from src.data_processing.processor import NetworkDataProcessor

processor = NetworkDataProcessor()
processed = processor.parse_cicids2017_format(data)

X, y = processor.prepare_features_for_ml(processed)
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
print("Feature columns:", list(X.columns[:10]), "...")

INFO:src.data_processing.processor:Parsing CICIDS2017 format...
INFO:src.data_processing.processor:Successfully parsed CICIDS2017 format. Shape: (2000, 53)
INFO:src.data_processing.processor:Preparing features for machine learning...
INFO:src.data_processing.processor:Prepared 49 features for ML


Features: 49, Samples: 2000
Feature columns: ['src_port', 'dst_port', 'packet_length', 'duration', 'flow_bytes_sent', 'flow_bytes_received', 'packets_sent', 'packets_received', 'duration_log', 'duration_sqrt'] ...


## 4. Train anomaly detection model

In [11]:
from src.models.anomaly_detector import AnomalyDetector

detector = AnomalyDetector(models_dir=str(ROOT / 'models'))
results = detector.train_isolation_forest(X, contamination=0.05)

print(f"Model: {results.get('model', 'N/A')}")
print(f"Anomalies detected: {results.get('anomaly_count', 0)}")
print(f"Anomaly rate: {100*results.get('anomaly_rate', 0):.1f}%")

INFO:src.models.anomaly_detector:Training Isolation Forest model...
INFO:src.models.anomaly_detector:Isolation Forest trained. Detected 100 anomalies


Model: Isolation Forest
Anomalies detected: 100
Anomaly rate: 5.0%


## 5. Visualize traffic and anomalies

In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Protocol distribution (pie)
protocol_counts = data['protocol'].value_counts()
fig1 = px.pie(values=protocol_counts.values, names=protocol_counts.index, title='Protocol distribution')
fig1.show()

In [13]:
# Packet length distribution (normal vs anomaly)
data['type'] = data['is_anomaly'].map({0: 'Normal', 1: 'Anomaly'})
fig2 = px.histogram(data, x='packet_length', color='type', nbins=40, title='Packet length by type')
fig2.show()

In [14]:
# Anomaly scores (if available from Isolation Forest)
if results and 'anomaly_scores' in results:
    scores = results['anomaly_scores']
    preds = results['predictions']
    fig3 = go.Figure()
    fig3.add_trace(go.Histogram(x=scores, name='Anomaly score', nbinsx=50))
    fig3.update_layout(title='Isolation Forest anomaly scores', xaxis_title='Score')
    fig3.show()

## Interactive graph, histogram & heatmap

The three charts below are **interactive** (zoom, pan, hover). When you run this notebook on Colab or locally, use the toolbar to explore.

In [15]:
# --- INTERACTIVE HISTOGRAM: packet length & anomaly scores ---
fig_hist = make_subplots(rows=1, cols=2, subplot_titles=('Packet length distribution', 'Anomaly scores'))
fig_hist.add_trace(go.Histogram(x=data['packet_length'], nbinsx=40, name='Packet length', marker_color='steelblue'), row=1, col=1)
if results and 'anomaly_scores' in results:
    fig_hist.add_trace(go.Histogram(x=results['anomaly_scores'], nbinsx=50, name='Anomaly score', marker_color='coral'), row=1, col=2)
fig_hist.update_layout(title_text='Interactive histogram', showlegend=True, height=400)
fig_hist.show()

In [16]:
# --- INTERACTIVE HEATMAP: feature correlation ---
corr = X.corr()
# Limit to first 15 features for readability
cols = X.columns[:15].tolist()
corr_sub = X[cols].corr()
fig_heat = go.Figure(data=go.Heatmap(z=corr_sub.values, x=cols, y=cols, colorscale='RdBu', zmid=0))
fig_heat.update_layout(title='Interactive heatmap: feature correlations', height=500, xaxis_tickangle=-45)
fig_heat.show()

In [17]:
# --- INTERACTIVE GRAPH: traffic over time (packets per bucket) ---
# Build time buckets from row index (simulated time)
bucket_size = max(1, len(data) // 50)
data_sorted = data.reset_index(drop=True)
data_sorted['bucket'] = (data_sorted.index // bucket_size) * bucket_size
time_agg = data_sorted.groupby('bucket').agg(
    packets=('packet_length', 'count'),
    anomalies=('is_anomaly', 'sum'),
    avg_len=('packet_length', 'mean')
).reset_index()
fig_graph = go.Figure()
fig_graph.add_trace(go.Scatter(x=time_agg['bucket'], y=time_agg['packets'], mode='lines+markers', name='Packets'))
fig_graph.add_trace(go.Scatter(x=time_agg['bucket'], y=time_agg['anomalies'], mode='lines+markers', name='Anomalies'))
fig_graph.update_layout(title='Interactive graph: traffic volume over time', xaxis_title='Sample bucket', yaxis_title='Count', height=400, hovermode='x unified')
fig_graph.show()

## 6. Summary

You've run the core pipeline: **synthetic data → processing → Isolation Forest training → visualizations.**

- **Full app:** Clone the repo and run `streamlit run src/dashboard/app.py` for the full dashboard.
- **Train all models:** `python trafficanomaly.py train`
- **Run tests:** `python trafficanomaly.py test`